# PortoFlow — 10×10 km, toàn nguồn dữ liệu

Hai Inputs: **code Dataset từ portoflow_kaggle.zip** và **Porto train.csv.zip/train.csv đầy đủ**.
CPU/None, Internet On. Save Version → Save & Run All. Gói độc lập có sẵn graph đã kiểm.

Tâm Porto: [-8.63, 41.16]; vuông 10.000×10.000m trong EPSG:32629, đệm matching500m.
Graph drive tải mới; cap60.000cạnh context, không phải số cạnh được hứa trước.
Quét đến EOF (cap2triệu dòng); split toàn chuyến trước cutoff2014-05-01UTC.
Giữ mọi block liên tục≥6GPS trong context chạm vùng nghiên cứu; không cần90%GPS trong vùng.
Giữ gapQC, trim đầu/cuối block và support3chuyến/2taxi/2ngày, ratio0.7/0.4.
Evidence chỉ lấy phần cạnh bên trong100km². Tất cả train/holdout đủ điều kiện được matching;
holdout không vào profile. Không dừng ở80%; unknown giữ nguyên.

Trần mỗi phiên: graph10phút; scan90phút; matching6giờ; report30phút; RAM12.000MiB/output12GiB.
Setup/probes có cap riêng trong code. Đây là trần, chưa là dự báo thời gian toàn nguồn.
Checkpoint sau20parent. Hết phiên giữ toàn bộ Output; attach lại và đặt RESUME tới thư mục
portoflow_run để tiếp tục. Mã và dependency versions phải khớp; khác identity sẽ dừng.

Output nhỏ: portoflow_review.zip. **Giữ toàn bộ Output**, nhất là source_index.sqlite,
graph/, context_graph/, region_batches/ và code_revisions.json. Không cần nén lại toàn checkpoint.
review_export chỉ là candidate; metric ETA trên các component holdout và overlays cần review trước khi dùng trong app.
Notebook tạo candidate riêng, không tự kích hoạt app. Chưa có kết quả Porto thật cho lượt này.


In [ ]:

from pathlib import Path, PurePosixPath
import hashlib, json, os, shutil, stat, subprocess, sys, time, traceback, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/portoflow_project')
WORK = Path('/kaggle/working/portoflow_run')
WORK.mkdir(parents=True, exist_ok=True)
EXPECTED = {'README.md': 'c646ec840faa3ec7700022b2f4e12ec8c1eb5803ed70e641ce62079c8a1458f6', 'app/journey_routing.py': 'e8eacecf71ba09d8d6f9fccbaded80f4185784437eaba3233b154ab357b41cd9', 'app/journey_view.py': '3f0e46891e4f229f34ef278ad8ae485d6370fbedd065427e71f1010591d0fde1', 'app/map_frontend/index.html': 'ec6b534c65e81d60afc6b209da9fbaf3c0cb85f28b349cc9f0269f159833806d', 'app/map_frontend/map.js': '5ce36fc16638ff1c3515e66ff98a43491243a14e3caa3ee543185dd0e96dfb07', 'app/persistent_map.py': '23673e833bd9c6c6adae83b29d5f03747c777d2f5530bcd63ef9873c14d24b8d', 'app/road_display.py': '406b7dabf63cc64a141dc205f69793c6887913333847eefdceda0b403f2d1132', 'app/streamlit_app.py': '4070105698bf34d1c27e4f764dc48c93111e4f3eeac9ed51993e5d64470f4024', 'app/ui_logic.py': 'c15b6df0ba36346105ad9125ea1cc7c4331cfd38dd2c8dfb818e167cfffb52ce', 'configs/default.yaml': '87e67d6d2e606cd9679d2b818cfc93f3722dfe168efb966c31102c427533115a', 'configs/study_area.yaml': '3898f37ece0e204ddf94b521cc4b6850fe4161db4d9f29bf064a43de964984e3', 'docs/PIPELINE.md': '76a1f87b100030294befa441415c2d81597067ed5ed783bcb5ea71cd581eec34', 'pyproject.toml': 'f82189432fa9d01a24defe460f8c99696972bb289698d681eab9fd82570867d4', 'requirements/constraints.txt': '461ae5985f94aae9d02d5da3552ff09e4f18eed578c5334dfecee65d90ca6537', 'scripts/launch_app.py': '15f6a9b5b559cc16261ec50ab8e5ff79eb1a3e6b5b4656b43ec05bbb2aa373b5', 'scripts/probe_fastmm_native.py': 'f1fa3ded2c8dff95b72e8609e9900ab8fb500db1cd77278ea72ea47ceca09a79', 'scripts/report_recovery.py': 'c24d11ad11917b2bee742e8eb114aaf51b42c69cc2127ec4c355f5a1eb254263', 'scripts/run_pipeline.py': 'bf2a31813722edf870ca14deb7b8e3a9db86d69574f6ad4dec0a2ce5f2bc05e8', 'src/portoflow/__init__.py': 'ce68b7aaf2f0f077063031d7206e185230d57ff852c2af2c802d959aa6de955a', 'src/portoflow/config.py': '6e1963fdbe6ecd5e8d7420000e3a3c30e4dd72b1d53676b1fdb9e0c375c1d8cc', 'src/portoflow/congestion.py': '58930df59d669afb14154f8d1ed65d4e7c0fa1a52bf4c02ba68091f26bda14d4', 'src/portoflow/contracts.py': 'ef321f5829d847bbd8f704ac847f01cda9ded4fa1860d390a65fb5d5459e7ee2', 'src/portoflow/data.py': '889caafaeb02e137325a799b996e18779d6f4743eca88d8ebcf228a4f47ee762', 'src/portoflow/export.py': '951348ce1383c6e8e750bb829c60d29d8d147e7778ff01a7af6c8c33b012e55b', 'src/portoflow/graph.py': 'ea5b2597ae17ef57f4393863bf1cf4e1486ee6da1c9fdd6a789a8ba53e829bb7', 'src/portoflow/matching.py': 'df5780da53e57564abd4d451fe698c001f08b82a83a12f0efa90e8766b9a95d3', 'src/portoflow/pipeline.py': 'eb187fc2de9a751ddd4af910a784a16c93e7d8f1eb4cf0166e4a9c81d836136c', 'src/portoflow/preprocess.py': 'd7ec5c19cce02d8ea3e7d415333bec42fc25edd0242d5bb468a59449e534db5d', 'src/portoflow/profiles.py': 'f282d3f5fb11ef8910caf76df19120e8ffb4cf9cf4199d149b5d46ef94bb2c93', 'src/portoflow/route.py': '27729173359cf2f32d0f320dc40e02783bcb3c331b99e5b3e33c416f0fb0512a', 'src/portoflow/schema_v1.json': '28b20399945a89ecc930339b4572d1681604cd61ec3052ceae74ded6bdac9b42', 'src/portoflow/schemas.py': '7696d61e82bebaac381960d4c66aa8d15d44a84177ec2c666b7e05f531279b16', 'src/portoflow/segment_qc.py': 'e2613fb6c79b76ff78fd7f2c1e920731829cbbb1fa24b4aadd48b6a410032268', 'src/portoflow/speed.py': '5bd01482a89cf61bbe2723874a5d272724b534eb898acf9f97661b74c0144ede', 'src/portoflow/state.py': '6d80e53ce56074b66dbde98c054328fe277205bfab3d5785914435c46ada405d', 'src/portoflow/storage.py': 'e937952c364a15af437f58e0007662b786c29d91ab990728f572a148197ac07d', 'src/portoflow/synthetic.py': '57f8dac3bc881d077ffe213d5ec753b4411a0e23f082993b078a29e35a53081a', 'tests/test_pipeline.py': '4ce9be2697a3d35cbca9fa30f3b247124130b0aabe094ee364e92fa18727d5f0', 'resources/graph/graph_metadata.json': 'b5335fda9c714819d48787d1b88089a1b4871783c88000a413b4be05cde6b467', 'resources/graph/road_edges.parquet': 'e16e402d6879d357e946ff0239a0015ced865a53f38b303eaf8886bed5a0ba98', 'resources/graph/road_graph.graphml': '0b64ad347d65a365159b97261ce28958091e6cfccbbd4cb617754cad25393f2c', 'resources/context_graph/graph_metadata.json': 'f59e61532fdfd6107f33d589243081ff6c81ca9c47b908542b425278d88ad547', 'resources/context_graph/road_edges.parquet': '3a0af3181eff11323b6e94144f714c7c8ef209692e368caed4e44d9d49a30b69', 'resources/context_graph/road_graph.graphml': '3630f79f89f3d5eed84b4d7071dc1aca64a943fc01331471940e20f78b38980d', 'resources/region.json': '3611716eecf29db8fab00c0e0dfae0d73cfe6e74d9a48d09f6d03a9cadf80763'}
SOURCE = None  # Set an explicit Path here only when more than one train source is attached.
RESUME = None  # Optional path to an attached portoflow checkpoint DIRECTORY.
PHASES = ['graph', 'scan', 'match', 'report']
TOKEN = 'PORTOFLOW_FULL_SOURCE_V1'

def copy_code():
    roots = [p.parent.parent for p in INPUT.rglob('run_pipeline.py')
             if p.parent.name == 'scripts' and (p.parent.parent/'pyproject.toml').is_file()]
    for root in roots:
        if all((root/n).is_file() and hashlib.sha256((root/n).read_bytes()).hexdigest()==h for n,h in EXPECTED.items()):
            for relative in EXPECTED:
                dest = PROJECT/relative; dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(root/relative, dest)
            return
    archives = list(INPUT.rglob('portoflow_kaggle.zip'))
    if len(archives)!=1:
        raise ValueError('Attach the matching portoflow_code Dataset')
    with zipfile.ZipFile(archives[0]) as archive:
        for relative, checksum in EXPECTED.items():
            data = archive.read('portoflow_project/'+relative)
            if hashlib.sha256(data).hexdigest()!=checksum:
                raise ValueError('Code checksum differs: '+relative)
            dest = PROJECT/relative; dest.parent.mkdir(parents=True, exist_ok=True); dest.write_bytes(data)

def restore():
    if RESUME is None:
        return
    root = Path(RESUME).resolve()
    if not (root/'source_index.sqlite').is_file() or not (root/'region.json').is_file():
        raise ValueError('RESUME must point to the saved run directory, not an old07 checkpoint')
    if (WORK/'region.json').exists():
        raise ValueError('Do not merge two run directories; use the current WORK or a fresh kernel')
    shutil.copytree(root, WORK, dirs_exist_ok=True)

def run(command, name, seconds):
    logs = WORK/'logs'; logs.mkdir(exist_ok=True)
    print('START', name, '| cap seconds:', seconds, flush=True)
    with (logs/name).open('w') as stream:
        process = subprocess.Popen(list(map(str, command)), cwd=PROJECT, env=environment,
            stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        start = time.monotonic(); offset = 0
        try:
            while process.poll() is None:
                time.sleep(5)
                with (logs/name).open(errors='replace') as reader:
                    reader.seek(offset); text = reader.read(); offset = reader.tell()
                if text:
                    print(text[-12000:], end='', flush=True)
                else:
                    print(name, '| still running |', round(time.monotonic()-start), 's', flush=True)
                if time.monotonic()-start>seconds:
                    raise TimeoutError(name+' deadline; committed checkpoints retained')
            with (logs/name).open(errors='replace') as reader:
                reader.seek(offset); print(reader.read()[-12000:], flush=True)
            if process.returncode:
                raise RuntimeError(name+' failed; inspect its log and retained Output')
        finally:
            if process.poll() is None:
                import signal
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()

def review_bundle():
    # Full checkpoints remain as ordinary run-directory files in saved Kaggle Output.
    destination = Path('/kaggle/working/portoflow_review.zip')
    with zipfile.ZipFile(destination, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in WORK.rglob('*'):
            relative = path.relative_to(WORK)
            if not path.is_file() or path.stat().st_size>16*1024**2:
                continue
            if relative.parts[0]=='portoflow_recovery':
                if len(relative.parts)>1 and relative.parts[1] in {'profile_cache','evaluation_cache','evaluation_scratch','graph','matplotlib_cache'}:
                    continue
                if len(relative.parts)>1 and relative.parts[1]=='candidate_export':
                    if path.name in {'congestion_profiles.parquet','pipeline_config.json','diagnostics.png','run_metadata.json'}:
                        archive.write(path,relative.as_posix())
                elif path.suffix in {'.json','.txt'}:
                    archive.write(path,relative.as_posix())
                continue
            if relative.parts[0]=='region_batches':
                if path.name=='review_cases.json' and path.stat().st_size>2:
                    archive.write(path, relative.as_posix())
                continue
            if relative.parts[0] in {'native_cache','osm_cache','region_evidence.sqlite','source_index.sqlite','review_export','test_tmp'}:
                continue
            if path.suffix in {'.json','.txt','.yaml'}:
                archive.write(path, relative.as_posix())
    print('SEND FOR REVIEW:', destination, flush=True)
    print('RETAIN FULL Output directory:', WORK, '| includes checkpoints and candidate export', flush=True)

environment = dict(os.environ, PYTHONPATH=str(PROJECT/'src'), PYTHONUNBUFFERED='1',
    PYTHONFAULTHANDLER='1', MPLBACKEND='Agg', MPLCONFIGDIR=str(WORK/'matplotlib_cache'),
    UV_PYTHON_INSTALL_DIR='/tmp/portoflow_python', UV_CACHE_DIR='/tmp/portoflow_uv',
    OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
try:
    copy_code(); restore()
    resource = PROJECT/'resources'
    if not (WORK/'region.json').exists() and (resource/'region.json').is_file():
        for name in ('graph','context_graph'):
            shutil.copytree(resource/name,WORK/name,dirs_exist_ok=True)
        shutil.copyfile(resource/'region.json',WORK/'region.json')
    (WORK/'code_revisions.json').write_text(json.dumps(EXPECTED, indent=2))
    if SOURCE is None:
        candidates = list(INPUT.rglob('train.csv.zip'))
        if not candidates:
            candidates = list(INPUT.rglob('train.csv'))
        if len(candidates)!=1:
            raise ValueError('Attach full Porto train.csv.zip/train.csv; select SOURCE explicitly if ambiguous')
        SOURCE = candidates[0]
    print('Full source:', SOURCE, '| 10 km metric square | all eligible train/holdout | unknown retained', flush=True)
except Exception:
    (WORK/'BLOCKER.txt').write_text(traceback.format_exc()); review_bundle(); raise


In [ ]:

try:
    run([sys.executable,'-m','pip','install','uv>=0.8,<1'], 'setup_uv.txt', 180)
    uv = shutil.which('uv')
    if not uv:
        raise RuntimeError('uv executable unavailable')
    run([uv,'python','install','3.12'], 'setup_python.txt', 600)
    venv = Path('/tmp/portoflow_venv')
    if not (venv/'bin/python').is_file():
        run([uv,'venv','--python','3.12',venv], 'setup_venv.txt', 120)
    python = venv/'bin/python'
    freeze = WORK/'logs/environment_freeze.txt'
    install = [uv,'pip','install','--python',python,'--only-binary','fastmm']
    if RESUME is not None and freeze.is_file():
        install += ['-r',freeze]
    else:
        install += ['-c',PROJECT/'requirements/constraints.txt','-e',str(PROJECT)+'[test]','fastmm==0.3.2']
    run(install, 'setup_dependencies.txt', 900)
    run([uv,'pip','freeze','--python',python], 'environment_freeze.txt', 60)
    run([python,'-m','pytest','tests/test_pipeline.py','-q','--basetemp',WORK/'test_tmp'], 'synthetic_tests.txt', 180)
    run([python,PROJECT/'scripts/probe_fastmm_native.py','--output',WORK/'native_probe'], 'native_probe.txt', 120)
    caps = {'graph':600, 'scan':5460, 'match':21660, 'report':1860}
    for phase in PHASES:
        command = [python,PROJECT/'scripts/run_pipeline.py','--phase',phase,'--work',WORK,
                   '--plan',PROJECT/'configs/study_area.yaml','--config',PROJECT/'configs/default.yaml',
                   '--authorize-run',TOKEN]
        if phase=='scan':
            command += ['--source',SOURCE]
        run(command,phase+'.txt',caps[phase])
    print(json.loads((WORK/'result_summary.json').read_text())['status'], flush=True)
finally:
    review_bundle()
